In [ ]:

from typing import Any, Dict, List, Optional

from pydantic import BaseModel

from agente_avaliacao_imagens.schemas import AnaliseImagens, FeedbackImagens


class ReActInput(BaseModel):
    """Entrada do agente ReAct de análise de imagens."""

    fotos_urls: List[str] = []
    api_key: Optional[str] = None

c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from phoenix.otel import register

tracer_provider = register(
  project_name="agente-react-imoveis",
  auto_instrument=True
)

08/05/2026 01:31:42 PM 📋 Ensuring phoenix working directory: C:\Users\jefer\.phoenix
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.schemas
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.tables
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.types
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.constraints
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.defaults
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.comments


OpenTelemetry Tracing Details
|  Phoenix Project: agente-react-imoveis
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\phoenix\otel\otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


In [3]:
import logging
from typing import List

from langchain_core.tools import tool

from agente_avaliacao_imagens.prompts import PROMPT_DESCREVER_FOTO
from agente_avaliacao_imagens.utils import processar_todos_lotes

logger = logging.getLogger(__name__)


@tool
async def descrever_fotos(fotos_urls: List[str]) -> str:
    """Processa as fotos do imóvel e devolve a descrição técnica de cada imagem.

    Use esta ferramenta para obter a descrição das fotos. Depois, com base nela,
    preencha a análise estruturada final (scores, problemas, pontos fortes).

    Args:
        fotos_urls: lista de URLs das fotos do imóvel.
    """
    if not fotos_urls:
        return "Nenhuma URL de foto fornecida."
    
    logger.info(f"Processando {len(fotos_urls)} fotos para descrição.")

    descricao = await processar_todos_lotes(fotos_urls, 5, prompt=PROMPT_DESCREVER_FOTO)
    if not descricao:
        logger.error("Nao foi possivel descrever as fotos.")
        return "Falha ao descrever as fotos."
    return descricao

In [14]:
import json
import logging
import os
from typing import List, Optional

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.prebuilt import create_react_agent

from agente_avaliacao_imagens.schemas import AnaliseImagens
#from .tools import descrever_fotos

logger = logging.getLogger(__name__)

MODELO_AGENTE = os.getenv("MODELO_AGENTE_IMAGENS",
                          #"z-ai/glm-5.2"#V
                          "meta/llama-3.1-8b-instruct"V
                          #"nvidia/nemotron-3-ultra-550b-a55b" #V
                          # "google/gemma-4-31b-it" X
                          #"poolside/laguna-xs-2.1" #X
                          #"qwen/qwen-2.5-72b-instruct" #X
                          ) #"moonshotai/kimi-k2.6")"deepseek-ai/deepseek-v4-flash")

SYSTEM_PROMPT = """Você é um engenheiro civil e especialista em avaliação de imóveis para house flipping.

Sua tarefa é analisar as fotos de um imóvel e produzir um relatório técnico estruturado.

Passos:
1. Chame a ferramenta `descrever_fotos` com as URLs das fotos. Ela devolverá a descrição técnica de cada imagem.
2. Analise a descrição recebida e preencha a análise estruturada final com os campos abaixo.

Definição de cada campo do relatório final:

- **score_conservacao (float 0-10):** condição geral de conservação/mainutenção do que está visível (infiltrações, trincas, desgaste, estado de paredes/teto).
- **score_acabamento (float 0-10):** qualidade/padrão dos materiais (piso, revestimentos, metais, portas, esquadrias).
- **score_potencial_reforma (float 0-10):** o quanto é viável/vantajoso reformar o espaço (nota alta = boa estrutura que valoriza com melhorias; nota baixa = exige demolição pesada ou já está em ótimo estado).
- **confianca_imagem (float 0-10):** o quanto a descrição é confiável, clara e útil para uma avaliação técnica.
- **imagem_aceitavel (bool):** `true` se a foto mostra elementos reais do imóvel e é clara; `false` se for irrelevante (selfie, parede escura, objeto aleatório) ou a descrição for vaga demais.
- **problemas_visiveis (List[str]):** patologias, defeitos, danos ou sinais de desgaste identificados. Vazio se não houver.
- **pontos_fortes (List[str]):** aspectos positivos observados (iluminação natural, piso em bom estado, acabamento moderno, área espaçosa). Vazio se não houver.
- **observacoes (str):** resumo da opinião técnica. Se `imagem_aceitavel = false` ou as notas forem baixas, use este campo para justificar tecnicamente.

Regras importantes:
- Baseie-se APENAS na descrição fornecida pela ferramenta. Nunca invente ou infira o que não está visível.
- Se um aspecto não puder ser avaliado, pondere as notas de forma neutra e registre a limitação em `observacoes`.
- Se as fotos não retratarem um ambiente de imóvel (imagem irrelevante/ilegível), marque `imagem_aceitavel = false`, atribua `0.0` a todos os scores e explique em `observacoes`.
- Responda SEMPRE em português.
"""


def criar_agente_imagens(api_key: Optional[str] = None):
    model = ChatNVIDIA(
        model=MODELO_AGENTE,
        api_key=api_key or os.getenv("NVIDIA_API_KEY"),
    )
    return create_react_agent(
        model=model,
        tools=[descrever_fotos],
        prompt=SYSTEM_PROMPT,
        response_format=AnaliseImagens,
    )


async def analisar_imagens(
    fotos_urls: List[str],
    api_key: Optional[str] = None,
) -> AnaliseImagens:
    """Executa o agente ReAct de análise de imagens e devolve o relatório estruturado."""
    agente = criar_agente_imagens(api_key=api_key)

    mensagem_usuario = {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Analise as fotos do imóvel:\n"
                    + json.dumps(fotos_urls, ensure_ascii=False, indent=2)
                ),
            }
        ]
    }

    resultado = await agente.ainvoke(mensagem_usuario)
    resposta = resultado.get("structured_response")

    if isinstance(resposta, dict):
        return AnaliseImagens(**resposta)
    return resposta

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1975982450.py, line 16)

In [1]:
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

In [ ]:
fotos = df['fotos'].iloc[6].tolist()

In [ ]:
fotos

['https://resizedimgs.zapimoveis.com.br/img/vr-listing/2be5cbec55da625c7063443f287d852b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/80f6aae4cdef3bb2baf1c8609934408b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/532e18b9c870fd26cfc2a712304896ab/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/ed7f7a3d250cc07765b1d97fa3e94448/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/fa37d92847ba04c856f07049f497b1c3/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zap

In [ ]:
response

AnaliseImagens(score_conservacao=8.0, score_acabamento=8.0, score_potencial_reforma=5.0, confianca_imagem=8.5, imagem_aceitavel=True, problemas_visiveis=[], pontos_fortes=[' Fachada de edifício', 'Bom estado de conservação', 'Design contemporâneo'], observacoes='A imagem é redundante, repetindo a fachada exibida nas fotos 1 e 5[20, 39]. Analisa apenas o exterior do edifício, que não apresenta danos visíveis[37]. Ausência de informações sobre os ambientes internos do imóvel[41].')

In [1]:
import pandas as pd
import json

In [16]:
#df = pd.read_json('olx_alugueis.json', lines=True)
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

#with open(PASTA_DADOS/ 'joinville_aluguel_olx_2026-08.json', 'r', encoding='utf-8') as f:
#df = json.load(f)

In [18]:
df_dados = pd.read_parquet(PASTA_DADOS /"joinville_imovelweb_2026-08.parquet")
#df = pd.read_csv("imoveis_joinville_50m2.csv")

In [19]:
df_dados['url'].to_dict()

{0: 'https://www.imovelweb.com.br/propriedades/flat-em-joinville-exclusivo-para-investimento-no-pool-2945254565.html?n_src=Listado&n_exp=bianca_lead_flow_rollout-original&n_pills=Portaria+24+horas&n_pg=1&n_pos=1&n_search_id=2aa2b02f-54c3-47f0-a1cb-aa154e9c83e5',
 1: 'https://www.imovelweb.com.br/propriedades/liberty-residence-3043719835.html?n_src=Listado&n_exp=bianca_lead_flow_rollout-original&n_pills=Ar+condicionado&n_pg=1&n_pos=3&n_search_id=2aa2b02f-54c3-47f0-a1cb-aa154e9c83e5',
 2: 'https://www.imovelweb.com.br/propriedades/apartamento-a-venda-em-joinville-sc-no-bairro-3009126946.html?duplicated=true&n_src=Listado&n_exp=bianca_lead_flow_rollout-original&n_pills=Varanda&n_pg=1&n_pos=2&n_search_id=2aa2b02f-54c3-47f0-a1cb-aa154e9c83e5',
 3: 'https://www.imovelweb.com.br/propriedades/urban-azaleia-apartamento-pirabeiraba-2994869504.html?n_src=Listado&n_exp=bianca_lead_flow_rollout-original&n_pg=1&n_pos=4&n_search_id=2aa2b02f-54c3-47f0-a1cb-aa154e9c83e5',
 4: 'https://www.imovelweb.com

In [20]:
df_dados

,url,titulo,metragem,quartos,suites,banheiros,vagas,idade,valor_imovel,condominio,...,bairro,cidade,uf,descricao,data_criacao,caracteristicas,fotos,lat,lng,fonte
0,https://www.imovelweb.com.br/propriedades/flat...,Flat em Joinville exclusivo para investimento ...,22 m²,1,1,1,None,26,R$ 250.000,None,...,Centro,Joinville,SC,Ibis Joinville*Imóvel para investimento (100% ...,2019-08-26T20:02:08Z,"[Acesso para deficientes, Câmeras de segurança...",[https://imgbr.imovelwebcdn.com/avisos/2/29/45...,-26.301310000000000,-48.849760000000003,imovelweb
1,https://www.imovelweb.com.br/propriedades/libe...,LIBERTY RESIDENCE,33 m²,1,None,None,None,Breve Lançamento,R$ 460.000,550,...,Anita Garibaldi,Joinville,SC,"Apartamento tipo Studio , 100% mobiliado ,à VE...",2026-08-25T06:12:55Z,[Andares: 3],[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.320067099999999,-48.852519499999999,imovelweb
2,https://www.imovelweb.com.br/propriedades/apar...,"Apartamento à venda em Joinville-SC, no bairro...",45 m²,1,1,1,1,3,R$ 345.620,None,...,Saguaçu,Joinville,SC,Apartamento Loft Giardino no Saguaçu - Residen...,2025-04-01T18:01:58Z,"[Bicicletário, Elevador, Salão de festas, Pisc...",[https://imgbr.imovelwebcdn.com/avisos/resize/...,-26.272374100000000,-48.837289099999999,imovelweb
3,https://www.imovelweb.com.br/propriedades/urba...,Urban Azaléia - Apartamento Pirabeiraba,None,None,None,None,None,None,None,None,...,Pirabeiraba (Pirabeiraba),Joinville,SC,"Apartamento Pirabeiraba - Urban Azaléia, 2 qua...",2024-04-02T14:22:23Z,"[Bicicletário, Churrasqueira (parrilla), Espaç...",[https://imgbr.imovelwebcdn.com/avisos/2/29/94...,-26.209039900000000,-48.908440700000000,imovelweb
4,https://www.imovelweb.com.br/propriedades/casa...,"Casa à venda em Joinville-SC, bairro Aventurei...",49 m²,2,None,1,2,1,R$ 360.000,None,...,Aventureiro,Joinville,SC,Geminado Casa Baixa à venda no Bairro Aventure...,2025-09-25T19:11:23Z,"[Área de serviço, Lavanderia]",[https://imgbr.imovelwebcdn.com/avisos/2/30/18...,-26.253569800000001,-48.802431700000006,imovelweb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,https://www.imovelweb.com.br/propriedades/apto...,Apto com 1 suite+2 quartos e duas vagas de gar...,None,2,1,1,2,None,R$ 670.000,None,...,América,Joinville,SC,Apartamento no Edifício Sense em JoinvilleApar...,2026-08-21T14:51:09Z,"[Elevador, Fitness/Sala de Ginástica, Piscina,...",[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.304375799999998,-48.846374399999987,imovelweb
656,https://www.imovelweb.com.br/propriedades/apar...,Apartamento Vila Nova - Porto Garten,43 m²,2,None,1,1,None,R$ 270.000,None,...,Vila Nova,Joinville,SC,Venha conhecer o mais novo empreendimento da V...,2026-08-21T14:51:10Z,[],[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.291948500000000,-48.905364099999999,imovelweb
657,https://www.imovelweb.com.br/propriedades/apar...,"Apartamento com 3 quartos, Anita Garibaldi - J...",None,3,1,2,1,None,R$ 620.000,None,...,Anita Garibaldi,Joinville,SC,"O apartamento contam com 3 dormitórios, sendo ...",2026-08-21T14:51:04Z,"[Churrasqueira (parrilla), Elevador, Fitness/S...",[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.314900999999998,-48.851621700000002,imovelweb
658,https://www.imovelweb.com.br/propriedades/casa...,"Casa Geminada com 2 quartos à Venda, Aventurei...",None,2,None,1,2,None,R$ 360.000,None,...,Aventureiro,Joinville,SC,Oportunidade imperdível para quem busca o prim...,2026-08-25T19:46:35Z,[],[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.253613300000001,-48.819134699999999,imovelweb


In [21]:
df_dados.isnull().sum()

url                  0
titulo               0
metragem           179
quartos             71
suites             462
banheiros          175
vagas              181
idade              351
valor_imovel        40
condominio         518
endereco             2
bairro               2
cidade               2
uf                   2
descricao            2
data_criacao         2
caracteristicas      0
fotos                0
lat                212
lng                212
fonte                0
dtype: int64

In [13]:
df_dados.drop_duplicates(subset=['url'])

,url,titulo,metragem,quartos,suites,banheiros,vagas,idade,valor_imovel,condominio,...,bairro,cidade,uf,descricao,data_criacao,caracteristicas,fotos,lat,lng,fonte
0,https://www.imovelweb.com.br/propriedades/loft...,"Loft com 1 dormitório à venda, 26 m² por R$ 29...",26 m²,1,None,None,1,None,R$ 296.000,None,...,Costa e Silva,Joinville,SC,"Loft com 1 dormitório à venda, 26 m² por R$ 29...",2026-03-27T07:22:54Z,"[Churrasqueira (parrilla), Elevador, Piscina]",[https://imgbr.imovelwebcdn.com/avisos/2/30/31...,None,None,imovelweb
1,https://www.imovelweb.com.br/propriedades/casa...,"Casa à venda em Joinville-SC, bairro Aventurei...",49 m²,2,None,1,2,1,R$ 360.000,None,...,Aventureiro,Joinville,SC,Geminado Casa Baixa à venda no Bairro Aventure...,2025-09-25T19:11:23Z,"[Área de serviço, Lavanderia]",[https://imgbr.imovelwebcdn.com/avisos/2/30/18...,-26.253569800000001,-48.802431700000006,imovelweb
2,https://www.imovelweb.com.br/propriedades/vibe...,Vibe 1285 - Apartamento à venda no bairro Buca...,48 m²,2,1,2,1,Breve Lançamento,R$ 460.000,None,...,Bucarein,Joinville,SC,Apartamento Giardino com Suíte Master no VIBE ...,2026-05-16T03:07:08Z,"[Churrasqueira (parrilla), Elevador, Fitness/S...",[https://imgbr.imovelwebcdn.com/avisos/2/30/35...,-26.315383700000001,-48.844377799999996,imovelweb
3,https://www.imovelweb.com.br/propriedades/alge...,"Álgebra - Apartamento em Bom Retiro, Joinville/SC",49 m²,1,1,1,None,7,R$ 195.000,225,...,Bom Retiro,Joinville,SC,"Está procurando um imóvel compacto, funcional ...",2026-02-05T04:10:02Z,"[Área de serviço, Lavanderia, Permite animais,...",[https://imgbr.imovelwebcdn.com/avisos/2/30/27...,-26.259483899999999,-48.847349399999998,imovelweb
4,https://www.imovelweb.com.br/propriedades/loft...,"Loft com 1 dormitório à venda, 21 m² por R$ 30...",22 m²,1,None,None,1,None,R$ 300.000,None,...,Glória,Joinville,SC,"Loft à venda, 21 m² por R$ 300.000 - Glória - ...",2026-07-28T07:21:48Z,"[Brinquedoteca, Churrasqueira (parrilla), Elev...",[https://imgbr.imovelwebcdn.com/avisos/2/30/41...,None,None,imovelweb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,https://www.imovelweb.com.br/propriedades/apto...,Apto com 1 suite+2 quartos e duas vagas de gar...,None,2,1,1,2,None,R$ 670.000,None,...,América,Joinville,SC,Apartamento no Edifício Sense em JoinvilleApar...,2026-08-21T14:51:09Z,"[Elevador, Fitness/Sala de Ginástica, Piscina,...",[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.304375799999998,-48.846374399999987,imovelweb
476,https://www.imovelweb.com.br/propriedades/apar...,"Apartamento com 3 quartos, Anita Garibaldi - J...",None,3,1,2,1,None,R$ 620.000,None,...,Anita Garibaldi,Joinville,SC,"O apartamento contam com 3 dormitórios, sendo ...",2026-08-21T14:51:04Z,"[Churrasqueira (parrilla), Elevador, Fitness/S...",[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.314900999999998,-48.851621700000002,imovelweb
477,https://www.imovelweb.com.br/propriedades/casa...,CASA NO CENTRO DE JOINVILLE,None,None,None,None,None,None,R$ 1.500.000,1,...,Bucarein,Joinville,SC,Ótima oportunidade de investimento em Joinvill...,2026-04-01T12:01:54Z,[],[https://imgbr.imovelwebcdn.com/avisos/2/30/31...,-26.311033250000001,-48.843925480000002,imovelweb
478,https://www.imovelweb.com.br/propriedades/apar...,"Apartamento com 2 Quartos à Venda, Adhemar Gar...",None,2,None,1,1,None,R$ 319.000,None,...,Adhemar Garcia,Joinville,SC,Novo empreendimento com apenas 6 unidades excl...,2026-08-21T14:51:13Z,"[Elevador, Fitness/Sala de Ginástica]",[https://imgbr.imovelwebcdn.com/avisos/2/30/43...,-26.323794899999999,-48.806124199999999,imovelweb
